# Collected edits, OOD generality

All three dermatologists go into one codebook per model, one row per case. A case marked wrong by anyone is edited; if the corrections disagree it is printed and dropped. Reliability and locality stay in the perturb_generality collect run. This notebook scores leftover combined MIDAS cases: saved top-1 against `y16`, then one edited generation against the same `y16`.

`gen_all before` is the saved prediction against `y16`. `gen_all after` is the edited generation against the same `y16`. `gen_acc` is how many originally wrong cases become correct. `gen_loc` is how many originally correct cases become wrong.

`dermedit` kernel. `OVERWRITE` recomputes a saved run. `RESUME` finishes a run that is still missing generality. With both off, a complete result is printed from disk and an unfinished one stays blank. One embedding mode and one `REJECT_THRESHOLD` per session. `run.py` runs both embedding modes at 1 and at 5.

In [1]:
import sys
from pathlib import Path

OOD = Path("/scratch/jq2uw/MME/derm_vlm_edit/editing/ood_generality")
sys.path.insert(0, str(OOD))
from utils.bootstrap import bootstrap, results_dir
bootstrap()

import config as C

USERS = [
    "jesse.y.liu.md@gmail.com",
    "achiou@stanford.edu",
    "lhunterellul",
]
MODELS = ["medgemma", "dermatollama"]
RATIONALE_MODE = C.RATIONALE_MODE
NORM_ANSWER = "y16"
SUBSAMPLE = 0
MAX_NEW_TOKENS = C.MAX_NEW_TOKENS
SEED = 42
OVERWRITE = False
RESUME = False

# EMBED_MODE:
#   standalone_dual  DINOv2 image concatenated with SBERT text. This is the current run.
#   dual_sbert       VLM vision activation concatenated with SBERT. Still dual.
#   vision           VLM vision activation only.
#   language         A middle language-layer activation only.
#   language_last    Late language-layer activation. The old VLM-key style.
EMBED_MODE = "standalone_dual"
# Percentile of key-key distances. 1 and 5 are the sweep. 0.1 is the renamed run.
REJECT_THRESHOLD = 1
IMAGE_ENCODER = "dinov2"
IMAGE_POOL = "cls"
TEXT_ENCODER = "sbert"
SCALER_DATASET = "coco"

OUT = results_dir(OOD, "collect", str(EMBED_MODE) + "_" + str(REJECT_THRESHOLD))
POOL = "all"
print("results:", OUT)

results: /scratch/jq2uw/MME/derm_vlm_edit/editing/ood_generality/results/standalone_dual/collect


In [2]:
import gc

from IPython.display import Markdown, display

from utils.edit import edit
from utils.edits import collect_all, collect_ood
from utils.eval import load_suite, load_vlm, needs_compute, render_table, run_suite

rows = []
for model_short in MODELS:
    if not needs_compute(OUT, POOL, model_short, overwrite=OVERWRITE, resume=RESUME):
        print(f"{POOL} / {model_short}: loaded")
        rows.append(load_suite(OUT, POOL, model_short))
        continue

    edits, report = collect_all(
        USERS, model_short, rationale_mode=RATIONALE_MODE,
        norm_answer=NORM_ANSWER, subsample=SUBSAMPLE,
    )
    print(f"{model_short}: {len(edits)} unique edits, {len(report['conflicts'])} conflicting cases dropped")
    for row in report["conflicts"]:
        users = ", ".join(str(u).split("@")[0] for u in row["users"])
        print(f"  drop {row['case_id']}: {users} -> {', '.join(row['answers'])}")
    ood = collect_ood(USERS, model_short, subsample=SUBSAMPLE)
    n_before_text = sum(1 for item in ood if item.get("pred_before"))
    print(f"{model_short}: {len(ood)} ood cases ({n_before_text} with a saved top-1)")
    if not edits:
        print(f"skip {model_short}")
        continue

    import torch
    vlm, config = load_vlm(
        model_short, max_new_tokens=MAX_NEW_TOKENS, seed=SEED,
        embed_mode=EMBED_MODE, image_encoder=IMAGE_ENCODER,
        image_pool=IMAGE_POOL, text_encoder=TEXT_ENCODER,
        scaler_dataset=SCALER_DATASET, reject_threshold=REJECT_THRESHOLD,
    )
    editor, queries = edit(vlm, config, edits)
    rows.append(run_suite(
        editor, vlm, queries, user=POOL, model_short=model_short,
        out_dir=OUT, ood_rows=ood,
        norm_answer=NORM_ANSWER, max_new_tokens=MAX_NEW_TOKENS,
        overwrite=OVERWRITE,
    ))
    del vlm, editor
    gc.collect()
    torch.cuda.empty_cache()

display(Markdown(render_table(rows)))

all / medgemma: loaded
all / dermatollama: loaded


| User | Model | gen_all before | gen_all after | gen_acc | gen_loc |
| --- | --- | --- | --- | --- | --- |
| all | MedGemma | 263/746 | 183/746 | 38/483 | 118/263 |
| all | DermatoLlama | 263/743 | 172/743 | 37/480 | 128/263 |